In [15]:
from dotenv import load_dotenv
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [16]:
load_dotenv()

True

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [18]:
llm.invoke("Explain me EDA in 2 lines")

AIMessage(content='Exploratory Data Analysis (EDA) is the process of analyzing and visualizing datasets to summarize their main characteristics, often using statistical graphics and plots. It helps identify patterns, trends, and anomalies, guiding further analysis and model development.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 15, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_50880c66bf', 'id': 'chatcmpl-Dhr9kBFVqOoT98gLK1OYHWKzaN2dq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4937-1849-7ed3-95f6-ee87c67400df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_token

In [19]:
pdf_reader = PyPDFLoader('../content/sample.pdf')

documents = pdf_reader.load()

documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-21T11:06:22+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-21T11:06:22+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../content/sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Introduction to Retrieval-Augmented Generation\n (RAG)\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large\nlanguage model. Instead of relying only on the model's training data, RAG fetches relevant context from\nan external knowledge base and provides it to the model at inference time.\nWhy use RAG?\nRAG reduces hallucination, allows the model to use up-to-date information, and makes it possible to\ncite sources. It is much cheaper than fine-tuning a model on private data and the knowledge base can\nbe updated without retraining.\nCore

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(documents)

chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-21T11:06:22+05:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-21T11:06:22+05:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../content/sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Introduction to Retrieval-Augmented Generation\n (RAG)\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large\nlanguage model. Instead of relying only on the model's training data, RAG fetches relevant context from\nan external knowledge base and provides it to the model at inference time.\nWhy use RAG?\nRAG reduces hallucination, allows the model to use up-to-date information, and makes it possible to"),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-21T11:06:22

In [21]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

# FAISS: Facebook AI similary search
db = FAISS.from_documents(documents=chunks, embedding=embeddings)


In [22]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.prompts import PromptTemplate


CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template("""Given the following converstiond and a follow up questinns, repharse the following 
    Chat History :
    {chat_history}                                                                                          
    Follow up Input: {question}                                                    
    Standalone questions: """)

In [23]:
qa = ConversationalRetrievalChain.from_llm(llm=llm, retriever=db.as_retriever(), condense_question_prompt=CONDENSE_QUESTION_PROMPT, return_source_documents=True, verbose=True)


In [24]:
chat_history=[]

query = """What is Rag"""

result = qa.invoke({"question": query, "chat_history": chat_history})

print(result["answer"])



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Introduction to Retrieval-Augmented Generation
 (RAG)
What is RAG?
Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large
language model. Instead of relying only on the model's training data, RAG fetches relevant context from
an external knowledge base and provides it to the model at inference time.
Why use RAG?
RAG reduces hallucination, allows the model to use up-to-date information, and makes it possible to

cite sources. It is much cheaper than fine-tuning a model on private data and the knowledge base can
be updated without retraining.
Core components
A RAG pipeline has four pieces: a document loader, a text splitter that creates chunks, an embedding
mod

---

## Modern LCEL version (industry standard)

The cells above use `ConversationalRetrievalChain` from `langchain_classic` — it works but is **deprecated**. Modern LangChain uses **LCEL** (LangChain Expression Language) where you compose pipelines with the `|` pipe operator (like Unix pipes).

### Legacy vs Modern

| Aspect | Legacy (`ConversationalRetrievalChain`) | Modern (LCEL with `\|`) |
|---|---|---|
| **Style** | Class-based, hidden orchestration | Functional, explicit composition |
| **Import** | `langchain_classic.chains` | `langchain_core.runnables` |
| **Composition** | `.from_llm(...)` factory | `retriever \| prompt \| llm \| parser` |
| **Streaming** | Not built-in | Native (`.stream()`) |
| **Async** | Manual | Native (`.ainvoke()`) |
| **Batching** | Manual | Native (`.batch()`) |
| **Debuggability** | Black box | Each step inspectable |
| **Deprecated?** | Yes (still works) | No (current standard) |

### Mental model

```
question → retriever → format docs → prompt → llm → parser → answer
```

Each `|` passes the output of the left side as input to the right side.

In [25]:
# --- Modern imports ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Retriever (reuse the same FAISS db from before)
retriever = db.as_retriever(search_kwargs={"k": 3})

# 2. Prompt
prompt = ChatPromptTemplate.from_template("""Answer the question using ONLY the context below.
If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {question}

Answer:""")

# 3. Helper to flatten retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Compose the chain with | pipes
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Run it (just a string in, just a string out — no dict gymnastics)
answer = rag_chain.invoke("What is RAG?")
print(answer)

Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a large language model. Instead of relying only on the model's training data, RAG fetches relevant context from an external knowledge base and provides it to the model at inference time.


### Bonus — LCEL also supports streaming

Same chain, words appear as the LLM generates them. Impossible with the legacy chain.

In [ ]:
for token in rag_chain.stream("What is RAG?"):
    print(token, end="", flush=True)

### Conversational version (with chat history)

If you want multi-turn chat (the user can ask follow-up questions that depend on earlier ones), use `create_history_aware_retriever` + `create_retrieval_chain`. This is the modern replacement for `ConversationalRetrievalChain`.

In [ ]:
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Step 1: Rewrite follow-up questions to standalone questions using chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest user question, "
               "rewrite the question so it can be understood without the history. "
               "Do NOT answer it — just rewrite, or return it as is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

# Step 2: Answer the (now standalone) question using retrieved docs
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context:\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
qa_chain = create_stuff_documents_chain(llm, qa_prompt)

# Step 3: Glue them together
conversational_rag = create_retrieval_chain(history_aware_retriever, qa_chain)

# --- Multi-turn demo ---
chat_history = []

q1 = "What is RAG?"
r1 = conversational_rag.invoke({"input": q1, "chat_history": chat_history})
print("Q1:", q1)
print("A1:", r1["answer"], "\n")

chat_history.extend([HumanMessage(content=q1), AIMessage(content=r1["answer"])])

q2 = "Why would I use it?"   # follow-up — only makes sense with history
r2 = conversational_rag.invoke({"input": q2, "chat_history": chat_history})
print("Q2:", q2)
print("A2:", r2["answer"])